# SyllabusSync — MVP
**TAC 459 — Generative AI & NLP | Team G**

Bryce Hamlet, Jonathan Fisher, Ethan Say, Boyue Dong, Stefanie Akilian, Lauren Graham, William Jou

---

## Overview
Students receive 4–5 syllabi every semester with inconsistent formatting and buried deadlines. **SyllabusSync** uses NLP techniques covered in TAC 459 to automatically extract every assignment, exam, and deadline from a syllabus into a clean, structured task list.

### NLP Techniques Used (from class demos)
| Technique | Source | Purpose |
|---|---|---|
| Named Entity Recognition (NER) | Demo 6.1 | Identify dates, organizations, and events in text |
| Regex pattern matching | Demo 6.1 | Detect assignment/exam keywords |
| `dateparser` | Demo 6 | Normalize inconsistent date formats |
| Sentence tokenization | Demo 6.1 | Split syllabus into processable units |

### Pipeline
```
Syllabus Text → Sentence Tokenization → NER (spaCy) → Keyword Detection → Date Extraction (dateparser) → Task List → CSV Export
```

In [ ]:
# Install required packages
!pip install spacy dateparser -q
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 103.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
#Imports
import spacy
import dateparser
import re
import pandas as pd
from datetime import datetime

# Load the SpaCy English model
nlp = spacy.load('en_core_web_sm')

print('spaCy version:', spacy.__version__)
print('Model loaded: en_core_web_sm')
print('Ready!')

spaCy version: 3.8.14
Model loaded: en_core_web_sm
Ready!


---
## Step 1 — Load Syllabus Text
Paste your syllabus text into the variable below, or upload a `.txt` file to Colab and read it in.

In [ ]:
#Option A: Paste Syllabus Text

SYLLABUS_TEXT = """
TAC 459 — Generative AI and NLP
Spring 2026 | Professor XXX

ASSIGNMENTS AND GRADING

Homework 1: Python & NLP Basics — Due January 20
Homework 2: Vector Databases — Due February 3
Homework 3: Word Embeddings (Word2Vec / GloVe) — Due February 17
Homework 4: Transformers — Due March 3
Homework 5: Fine-Tuning BERT — Due March 17
Homework 6: Syntax Analysis & NER — Due March 31
Homework 7: Sentiment Analysis — Due April 7
Homework 8: BERT Classification — Due April 14

Midterm Exam: February 24 — covers Weeks 1-5, in class
Final Exam: May 5, 2:00 PM — comprehensive, 3 hours

MVP Submittal: April 23 — submit via Canvas
Final Project Presentation: May 1 — in class, 10 minutes per team
Final Project Report: May 3 — submit via Canvas, PDF format

Weekly Reading Responses: Due every Sunday by 11:59 PM
Class Participation: Ongoing throughout semester
"""

# ── Option B: Read from uploaded .txt file ────────────────────────────────────
# with open('/content/syllabus.txt', 'r') as f:
#     SYLLABUS_TEXT = f.read()

print(f'Syllabus loaded: {len(SYLLABUS_TEXT)} characters, {len(SYLLABUS_TEXT.splitlines())} lines')

Syllabus loaded: 855 characters, 24 lines


---
## Step 2 — NER with spaCy (Demo 6.1)
We use spaCy's Named Entity Recognition pipeline to identify dates, events, and organizations in the syllabus — the same approach used in the Demo 6.1 NER section.

In [ ]:
#Run spaCy NER on the full syllabus

doc = nlp(SYLLABUS_TEXT)

# Extract all named entities
print('Named Entities found by spaCy:\n')
print(f'{"Entity":<35} {"Label":<12} {"Description"}')
print('-' * 70)

for ent in doc.ents:
    label_desc = spacy.explain(ent.label_) or ent.label_
    print(f'{ent.text:<35} {ent.label_:<12} {label_desc}')

print(f'\nTotal entities found: {len(doc.ents)}')

Named Entities found by spaCy:

Entity                              Label        Description
----------------------------------------------------------------------
NLP                                 ORG          Companies, agencies, institutions, etc.
Spring 2026                         DATE         Absolute or relative dates or periods
XXX                                 PERSON       People, including fictional
Python & NLP Basics                 ORG          Companies, agencies, institutions, etc.
January 20                          DATE         Absolute or relative dates or periods
February 3                          DATE         Absolute or relative dates or periods
February 17                         DATE         Absolute or relative dates or periods
March 3                             DATE         Absolute or relative dates or periods
March 17                            DATE         Absolute or relative dates or periods
Syntax Analysis & NER               ORG          Companies,

---
## Step 3 — Keyword Detection with Regex
NER alone doesn't catch everything — syllabi use inconsistent language. We combined spaCy NER with regex keyword matching to catch all assignment types.

In [ ]:
#Define keyword patterns for each task type
# Regex patterns to detect assignment types (case-insensitive)

TASK_PATTERNS = {
    'Exam':       r'\b(exam|midterm|final exam|quiz|test)\b',
    'Assignment': r'\b(homework|hw|assignment|problem set|pset|lab)\b',
    'Project':    r'\b(project|presentation|submittal|report|demo)\b',
    'Reading':    r'\b(reading|readings|read|chapter|article|paper|response)\b',
}

def detect_task_type(text):
    """Return the task type based on keyword matching."""
    text_lower = text.lower()
    for task_type, pattern in TASK_PATTERNS.items():
        if re.search(pattern, text_lower):
            return task_type
    return 'Other'

# Test the detector
test_lines = [
    'Homework 1: Python Basics — Due January 20',
    'Midterm Exam: February 24',
    'Final Project Presentation: May 1',
    'Weekly Reading Responses: Due every Sunday',
]

print('Keyword detection test:\n')
for line in test_lines:
    print(f'  "{line}"')
    print(f'  → Type: {detect_task_type(line)}\n')

Keyword detection test:

  "Homework 1: Python Basics — Due January 20"
  → Type: Assignment

  "Midterm Exam: February 24"
  → Type: Exam

  "Final Project Presentation: May 1"
  → Type: Project

  "Weekly Reading Responses: Due every Sunday"
  → Type: Reading



---
## Step 4 — Date Extraction with `dateparser`
`dateparser` handles the inconsistent date formats found in real syllabi: `January 20`, `Feb 3`, `02/24`, `May 5, 2:00 PM`, etc.

In [ ]:
#Extract and normalize dates

def extract_date(text):
    """
    Extract a date from a line of text using dateparser.
    Returns the date as a formatted string, or None if no date found.
    """
    # First try spaCy NER to find DATE entities in the text
    line_doc = nlp(text)
    date_entities = [ent.text for ent in line_doc.ents if ent.label_ == 'DATE']

    # If spaCy found a date entity, parse it with dateparser
    if date_entities:
        parsed = dateparser.parse(date_entities[0], settings={'PREFER_DAY_OF_MONTH': 'first'})
        if parsed:
            return parsed.strftime('%B %d, %Y')
        return date_entities[0]  # Return raw text if dateparser can't parse

    # Fallback: search the full line with dateparser
    # Look for common date patterns using regex first
    date_pattern = r'(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|'\
                   r'Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)'\
                   r'\s+\d{1,2}(?:,?\s*\d{4})?|\d{1,2}/\d{1,2}(?:/\d{2,4})?'
    match = re.search(date_pattern, text, re.IGNORECASE)
    if match:
        parsed = dateparser.parse(match.group(0))
        if parsed:
            return parsed.strftime('%B %d, %Y')
        return match.group(0)

    return None

# Test date extraction
test_dates = [
    'Homework 1: Python Basics — Due January 20',
    'Midterm Exam: February 24 — covers Weeks 1-5',
    'Final Exam: May 5, 2:00 PM — comprehensive',
    'Submit by 02/17 at midnight',
    'Weekly Reading Responses: Due every Sunday',
]

print('Date extraction test:\n')
for line in test_dates:
    print(f'  Input:  "{line}"')
    print(f'  Date:   {extract_date(line)}\n')

Date extraction test:

  Input:  "Homework 1: Python Basics — Due January 20"
  Date:   January 20, 2026

  Input:  "Midterm Exam: February 24 — covers Weeks 1-5"
  Date:   February 24, 2026

  Input:  "Final Exam: May 5, 2:00 PM — comprehensive"
  Date:   May 05, 2026

  Input:  "Submit by 02/17 at midnight"
  Date:   February 17, 2026

  Input:  "Weekly Reading Responses: Due every Sunday"
  Date:   None



---
## Step 5 — Full Extraction Pipeline
Now we combine everything: sentence tokenization → keyword detection → NER → date extraction → structured task list.

In [ ]:
#Full SyllabusSync extraction pipeline

def extract_tasks(syllabus_text):
    """
    Main pipeline: takes raw syllabus text and returns a list of task dicts.

    Steps:
    1. Sentence tokenization (spaCy)
    2. Filter sentences that contain task keywords
    3. Detect task type via regex
    4. Extract date via spaCy NER + dateparser
    5. Return structured task list
    """
    doc = nlp(syllabus_text)
    tasks = []

    # Also process line-by-line since syllabi use newlines heavily (not full sentences)
    lines = [line.strip() for line in syllabus_text.split('\n') if line.strip()]

    seen = set()  # Avoid duplicates

    for line in lines:
        # Skip very short lines (headers, section dividers)
        if len(line) < 10:
            continue

        # Check if this line mentions a task type
        task_type = detect_task_type(line)
        if task_type == 'Other':
            continue  # Skip lines that don't look like tasks

        # Extract date
        due_date = extract_date(line)

        # Clean up task name: remove date portion and "Due" prefixes
        task_name = re.sub(r'\s*[—–-]+\s*(Due\s*)?.*$', '', line, flags=re.IGNORECASE).strip()
        task_name = re.sub(r'\s*:.*?(due|by|on).*$', '', task_name, flags=re.IGNORECASE).strip()
        if not task_name:
            task_name = line[:60]  # Fallback: use first 60 chars

        # Deduplicate
        key = task_name.lower()[:40]
        if key in seen:
            continue
        seen.add(key)

        tasks.append({
            'Task': task_name,
            'Type': task_type,
            'Due Date': due_date or 'Not specified',
            'Raw Line': line,
        })

    return tasks


#Run the pipeline
tasks = extract_tasks(SYLLABUS_TEXT)

print(f'Extraction complete! Found {len(tasks)} tasks.\n')
print(f'{"Task":<45} {"Type":<12} {"Due Date"}')
print('-' * 80)
for t in tasks:
    print(f'{t["Task"]:<45} {t["Type"]:<12} {t["Due Date"]}')

Extraction complete! Found 14 tasks.

Task                                          Type         Due Date
--------------------------------------------------------------------------------
Homework 1                                    Assignment   January 20, 2026
Homework 2: Vector Databases                  Assignment   February 03, 2026
Homework 3: Word Embeddings (Word2Vec / GloVe) Assignment   February 17, 2026
Homework 4: Transformers                      Assignment   March 03, 2026
Homework 5: Fine                              Assignment   March 17, 2026
Homework 6: Syntax Analysis & NER             Assignment   March 31, 2026
Homework 7: Sentiment Analysis                Assignment   April 07, 2026
Homework 8                                    Assignment   April 14, 2026
Midterm Exam: February 24                     Exam         February 24, 2026
Final Exam: May 5, 2:00 PM                    Exam         May 05, 2026
MVP Submittal: April 23                       Project      Apri

---
## Step 6 — Results as a DataFrame

In [ ]:
#Display as a clean pandas DataFrame
df = pd.DataFrame(tasks)[['Task', 'Type', 'Due Date']]

# Summary stats
print('Task breakdown by type:')
print(df['Type'].value_counts().to_string())
print()

# Display full table
display(df)

Task breakdown by type:
Type
Assignment    8
Project       3
Exam          2
Reading       1



,Task,Type,Due Date
0,Homework 1,Assignment,"January 20, 2026"
1,Homework 2: Vector Databases,Assignment,"February 03, 2026"
2,Homework 3: Word Embeddings (Word2Vec / GloVe),Assignment,"February 17, 2026"
3,Homework 4: Transformers,Assignment,"March 03, 2026"
4,Homework 5: Fine,Assignment,"March 17, 2026"
5,Homework 6: Syntax Analysis & NER,Assignment,"March 31, 2026"
6,Homework 7: Sentiment Analysis,Assignment,"April 07, 2026"
7,Homework 8,Assignment,"April 14, 2026"
8,Midterm Exam: February 24,Exam,"February 24, 2026"
9,"Final Exam: May 5, 2:00 PM",Exam,"May 05, 2026"


---
## Step 7 — Export to CSV

In [ ]:
#Export to CSV

output_filename = 'syllabus_tasks.csv'
df.to_csv(output_filename, index=False)

print(f'Saved {len(df)} tasks to "{output_filename}"')
print()
print('Preview of CSV:')
print(df.to_csv(index=False))

from google.colab import files

df.to_csv('syllabus_tasks.csv', index=False)
files.download('syllabus_tasks.csv')
print(f'Downloaded {len(df)} tasks')

Saved 14 tasks to "syllabus_tasks.csv"

Preview of CSV:
Task,Type,Due Date
Homework 1,Assignment,"January 20, 2026"
Homework 2: Vector Databases,Assignment,"February 03, 2026"
Homework 3: Word Embeddings (Word2Vec / GloVe),Assignment,"February 17, 2026"
Homework 4: Transformers,Assignment,"March 03, 2026"
Homework 5: Fine,Assignment,"March 17, 2026"
Homework 6: Syntax Analysis & NER,Assignment,"March 31, 2026"
Homework 7: Sentiment Analysis,Assignment,"April 07, 2026"
Homework 8,Assignment,"April 14, 2026"
Midterm Exam: February 24,Exam,"February 24, 2026"
"Final Exam: May 5, 2:00 PM",Exam,"May 05, 2026"
MVP Submittal: April 23,Project,"April 23, 2026"
Final Project Presentation: May 1,Project,"May 01, 2026"
Final Project Report: May 3,Project,"May 03, 2026"
Weekly Reading Responses,Reading,"April 19, 2026"



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded 14 tasks


---
## Step 8 — NER Visualization
Visualize which parts of the syllabus spaCy recognized as named entities — using `displacy`, the same visualization tool from Demo 6.1.

In [ ]:
# Visualize NER with displacy
from spacy import displacy

# Run NER on first 500 characters so the visualization stays readable
sample_doc = nlp(SYLLABUS_TEXT[:500])

#Render
displacy.render(sample_doc, style='ent', jupyter=True)

---
#Summary

| Step | Technique | From Demo |
|---|---|---|
| Sentence tokenization | `spaCy` doc pipeline | Demo 6.1 |
| Named Entity Recognition | `spaCy` NER, `displacy` | Demo 6.1 |
| Keyword matching | `re` (regex) | Demo 6.1 |
| Date normalization | `dateparser` | Demo 6 |
| Structured output | `pandas` DataFrame | Throughout |
| CSV export | `pandas.to_csv()` | Throughout |

### Known Limitations
- spaCy's `en_core_web_sm` may misclassify some academic terms (same limitation noted in HW 6 audit)
- Date extraction accuracy depends on how consistently the syllabus formats dates
- Recurring deadlines (e.g. "every Sunday") are extracted as text but not expanded into a full calendar

### Phase 2 (Next Sprint)
- Google Calendar API sync via OAuth 2.0
- LLM layer (Claude API) for higher-accuracy extraction on messy syllabi
- Web UI built with Streamlit or React